# Preparação das instâncias municipais completas

Todas as etapas usam todos os setores do município, inclusive setores de demanda zero. Não há amostragem. As fontes e regras de demanda seguem a AED. As cinco maiores cidades de MG por população são comparadas com Lavras.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display
raiz = Path.cwd()
if raiz.name == "src":
    raiz = raiz.parent
saida = raiz / "outputs/grandes_municipios"
saida.mkdir(parents=True, exist_ok=True)

In [2]:
pop = pd.read_excel(raiz / "data/POP2026_20261029.xlsx", sheet_name="MUNICÍPIOS",
                    header=1, usecols=range(5), dtype="string")
pop.columns = ["uf", "codigo_uf", "codigo_municipio", "municipio", "populacao"]
pop["populacao"] = pd.to_numeric(
    pop.populacao.str.replace(r"\s*\(.*$", "", regex=True).str.replace(".", "", regex=False),
    errors="coerce")
ranking = pop.loc[pop.uf.eq("MG")].nlargest(5, "populacao").sort_values("populacao")
ranking["codigo"] = ranking.codigo_uf.str.zfill(2) + ranking.codigo_municipio.str.zfill(5)
ranking.to_csv(saida / "ranking_ibge_2026.csv", index=False)
municipios = {"3138203": "Lavras", **dict(zip(ranking.codigo, ranking.municipio))}
etapas = {"fundamental_iniciais": ("QT_MAT_FUND_AI", "QT_TUR_FUND_AI", "V01032"),
          "fundamental_finais": ("QT_MAT_FUND_AF", "QT_TUR_FUND_AF", "V01033"),
          "medio": ("QT_MAT_MED_PROP", "QT_TUR_MED_PROP", "V01034")}
assert all(c.startswith("31") for c in municipios)
fontes = next((raiz / "data").rglob("Tabela_Escola_2025_V2.csv")).parent

## Escolas elegíveis

Escolas públicas em atividade com matrículas positivas na etapa. A exclusão documentada do CENAV é mantida. Matrículas positivas sem informação de turmas precisam de revisão.

In [3]:
def ler_recorte(arquivo, colunas):
    blocos = pd.read_csv(arquivo, sep=";", encoding="latin1", dtype="string",
                         usecols=["CO_MUNICIPIO", *colunas], chunksize=20000)
    return pd.concat([b.loc[b.CO_MUNICIPIO.isin(municipios)] for b in blocos], ignore_index=True)
campos = ["CO_ENTIDADE", "NO_ENTIDADE", "TP_DEPENDENCIA", "TP_SITUACAO_FUNCIONAMENTO"]
escolas = ler_recorte(fontes / "Tabela_Escola_2025_V2.csv", campos)
for arquivo, indice in [("Tabela_Matricula_2025_V2.csv", 0), ("Tabela_Turma_2025_V2.csv", 1)]:
    colunas = ["CO_ENTIDADE", *[v[indice] for v in etapas.values()]]
    dados = ler_recorte(fontes / arquivo, colunas).drop(columns="CO_MUNICIPIO")
    escolas = escolas.merge(dados, on="CO_ENTIDADE", how="left", validate="one_to_one")
for campo in [v[i] for v in etapas.values() for i in [0, 1]]:
    escolas[campo] = pd.to_numeric(escolas[campo], errors="raise")

In [4]:
publica_ativa = escolas.TP_DEPENDENCIA.isin(["1", "2", "3"]) & escolas.TP_SITUACAO_FUNCIONAMENTO.eq("1")
escolas["exclusao"] = escolas.CO_ENTIDADE.map({"31347930": "Atendimento especializado CENAV"})
for etapa, (mat, tur, _) in etapas.items():
    escolas["elegivel_" + etapa] = publica_ativa & escolas[mat].gt(0) & escolas.exclusao.isna()
    pendente = escolas["elegivel_" + etapa] & escolas[tur].isna()
    assert not pendente.any(), f"Matrículas sem turmas em {etapa}"
escolas.to_csv(saida / "auditoria_escolas.csv", index=False)
display(escolas.groupby("CO_MUNICIPIO")[["elegivel_" + e for e in etapas]].sum())

,elegivel_fundamental_iniciais,elegivel_fundamental_finais,elegivel_medio
CO_MUNICIPIO,,,
3106200,288,266,133
3118601,81,80,43
3136702,115,100,36
3138203,20,22,8
3143302,87,71,43
3170206,93,91,36


## Todos os setores e grupos etários

Partes do mesmo setor são reunidas. Setores sem população permanecem com demanda zero. Valores etários suprimidos continuam ausentes na coluna original e têm uma estimativa separada.

In [5]:
codigos_sql = ",".join("'" + c + "'" for c in municipios)
malha = gpd.read_file(raiz / "data/BR_setores_CD2022.gpkg", layer="BR_setores_CD2022",
                      where=f"CD_MUN IN ({codigos_sql})",
                      columns=["CD_SETOR", "CD_MUN", "SITUACAO", "v0001"])
atributos = ["CD_MUN", "SITUACAO", "v0001"]
assert not malha.groupby("CD_SETOR")[atributos].nunique(dropna=False).gt(1).any().any()
assert malha.geometry.notna().all() and malha.geometry.is_valid.all()
malha = malha.dissolve(by="CD_SETOR", as_index=False)
assert malha.v0001.notna().all() and malha.v0001.ge(0).all()

In [6]:
colunas = ["CD_setor", "V01006", *[v[2] for v in etapas.values()]]
blocos = pd.read_csv(raiz / "data/Agregados_por_setores_demografia_BR.csv", sep=";",
                     dtype="string", usecols=colunas, chunksize=20000)
idades = pd.concat([b.loc[b.CD_setor.str[:7].isin(municipios)] for b in blocos], ignore_index=True)
idades = idades.rename(columns={"CD_setor": "CD_SETOR"})
assert idades.CD_SETOR.is_unique
for campo in colunas[1:]:
    valor = idades[campo].str.strip()
    idades[campo + "_suprimido"] = valor.eq("X").fillna(False)
    idades[campo] = pd.to_numeric(valor.mask(valor.eq("X")), errors="raise")
setores = malha.drop(columns="geometry").merge(idades, on="CD_SETOR", how="left", validate="one_to_one")
conhecidos = setores.V01006.notna()
assert setores.loc[conhecidos, "V01006"].eq(setores.loc[conhecidos, "v0001"]).all()

In [7]:
def distribuir_inteiros(pesos, total):
    assert np.isfinite(pesos).all() and pesos.ge(0).all() and pesos.sum() > 0
    valores = pesos.to_numpy(dtype=float) * total / pesos.sum()
    inteiros = np.floor(valores).astype(int)
    ordem = np.argsort(-(valores - inteiros), kind="stable")
    inteiros[ordem[:total - int(inteiros.sum())]] += 1
    assert inteiros.sum() == total
    return valores, inteiros

def estimar_idades(demanda, campo):
    conhecido = demanda[campo].notna()
    base = demanda.loc[conhecido, "v0001"].sum()
    assert base > 0
    taxa = demanda.loc[conhecido, campo].sum() / base
    return demanda[campo].astype(float).fillna(demanda.v0001 * taxa), taxa

## Coordenadas e cobertura

O catálogo é integrado pelo código INEP. As coordenadas devem ser numéricas e estar dentro do município. O complemento de rua já documentado de Lavras é reaproveitado. Complementos do Nominatim são lidos do arquivo auditável, sem novas consultas nesta preparação. Coordenadas originais a até 250 m da divisa são mantidas como aproximação sinalizada, conforme município administrativo do INEP. O ponto não é deslocado para dentro do limite.

In [8]:
catalogo = pd.read_csv(raiz / "data/Análise - Tabela da lista das escolas - Detalhado.csv",
                       dtype="string", usecols=["Código INEP", "Latitude", "Longitude", "Endereço"])
catalogo = catalogo.rename(columns={"Código INEP": "CO_ENTIDADE"})
assert catalogo.CO_ENTIDADE.is_unique
for eixo, campo in [("latitude", "Latitude"), ("longitude", "Longitude")]:
    catalogo[eixo] = pd.to_numeric(catalogo[campo].str.replace(",", ".", regex=False), errors="coerce")
coordenadas = escolas.merge(catalogo, on="CO_ENTIDADE", how="left", validate="one_to_one")
coordenadas["precisao"] = "catalogo_sem_precisao_informada"
coordenadas["fonte"] = "Catálogo INEP"

In [9]:
aprovadas = raiz / "outputs/3138203/escolas/geocodificacao_aprovada.csv"
if aprovadas.exists():
    for r in pd.read_csv(aprovadas, dtype={"CO_ENTIDADE": "string"}).itertuples():
        falta = coordenadas.latitude.isna() | coordenadas.longitude.isna()
        linha = coordenadas.CO_ENTIDADE.eq(r.CO_ENTIDADE) & falta
        coordenadas.loc[linha, ["latitude", "longitude"]] = [r.latitude, r.longitude]
        coordenadas.loc[linha, "precisao"] = r.precisao
        coordenadas.loc[linha, "fonte"] = r.fonte

In [10]:
complemento = raiz / "data/geocodificacao_aprovada_grandes_municipios.csv"
if complemento.exists():
    dados = pd.read_csv(complemento, dtype={"CO_ENTIDADE": "string"})
    assert dados.CO_ENTIDADE.is_unique
    for r in dados.itertuples():
        linha = coordenadas.CO_ENTIDADE.eq(r.CO_ENTIDADE)
        coordenadas.loc[linha, ["latitude", "longitude"]] = [r.latitude, r.longitude]
        coordenadas.loc[linha, "precisao"] = r.precisao
        coordenadas.loc[linha, "fonte"] = r.fonte
coordenadas["numerica_valida"] = (coordenadas.latitude.between(-90, 90) & coordenadas.longitude.between(-180, 180)).fillna(False)
coordenadas["dentro_municipio"] = False
coordenadas["repetida"] = False
coordenadas["distancia_limite_m"] = float("nan")

In [11]:
geometrias = {}
for codigo in municipios:
    recorte = malha.loc[malha.CD_MUN.eq(codigo)].copy()
    crs = recorte.estimate_utm_crs()
    recorte = recorte.to_crs(crs).set_index("CD_SETOR").sort_index()
    limite = recorte.geometry.union_all()
    linhas = coordenadas.CO_MUNICIPIO.eq(codigo) & coordenadas.numerica_valida
    pontos = gpd.GeoDataFrame(coordenadas.loc[linhas].copy(), geometry=gpd.points_from_xy(
        coordenadas.loc[linhas, "longitude"], coordenadas.loc[linhas, "latitude"]), crs=4326).to_crs(crs)
    coordenadas.loc[pontos.index, "dentro_municipio"] = pontos.geometry.map(limite.covers)
    coordenadas.loc[pontos.index, "distancia_limite_m"] = pontos.geometry.distance(limite)
    coordenadas.loc[pontos.index, "repetida"] = pontos.duplicated(["latitude", "longitude"], keep=False)
    geometrias[codigo] = (recorte, pontos.set_index("CO_ENTIDADE"))
coordenadas["proxima_divisa"] = (~coordenadas.dentro_municipio & coordenadas.distancia_limite_m.le(250)
                                  & coordenadas.fonte.eq("Catálogo INEP"))
coordenadas.loc[coordenadas.proxima_divisa, "precisao"] = "catalogo_proximo_divisa_ate_250m"
coordenadas["apta"] = coordenadas.numerica_valida & (coordenadas.dentro_municipio | coordenadas.proxima_divisa) & ~coordenadas.repetida
coordenadas.to_csv(saida / "auditoria_coordenadas.csv", index=False)

## Instâncias completas

A demanda calibrada mantém a soma das matrículas elegíveis. São salvos todos os setores, as capacidades e a matriz de distâncias de cada etapa. Todas as matrizes são materializadas quando as coordenadas estão completas.

In [12]:
def criar_demanda(codigo, etapa, total):
    campo = etapas[etapa][2]
    demanda = setores.loc[setores.CD_MUN.eq(codigo)].sort_values("CD_SETOR").copy()
    perfil, taxa = estimar_idades(demanda, campo)
    demanda["populacao_idade_estimada"] = perfil
    demanda["idade_imputada"] = demanda[campo].isna()
    demanda["demanda_proporcional"], demanda["demanda_inteira"] = distribuir_inteiros(perfil, total)
    return demanda, taxa

def distancias_completas(origens, destinos):
    dx = origens.x.to_numpy()[:, None] - destinos.x.to_numpy()[None, :]
    dy = origens.y.to_numpy()[:, None] - destinos.y.to_numpy()[None, :]
    return pd.DataFrame(np.hypot(dx, dy) / 1000, index=origens.index, columns=destinos.index)

In [13]:
def salvar_instancia(codigo, etapa, escola, demanda):
    pasta = saida / "instancias" / codigo / etapa
    pasta.mkdir(parents=True, exist_ok=True)
    escola.to_csv(pasta / "escolas.csv", index=False)
    demanda.to_csv(pasta / "demanda.csv", index=False)
    geometria, pontos = geometrias[codigo]
    if not escola.apta.all():
        return 0
    origens = geometria.geometry.centroid.reindex(demanda.CD_SETOR)
    destinos = pontos.geometry.reindex(escola.CO_ENTIDADE)
    D = distancias_completas(origens, destinos)
    assert D.shape == (len(demanda), len(escola)) and np.isfinite(D.to_numpy()).all()
    D.rename_axis("CD_SETOR").to_csv(pasta / "distancias_km.csv")
    return D.size

In [14]:
resumo = []
for codigo, municipio in municipios.items():
    for etapa, (campo, _, _) in etapas.items():
        escola = coordenadas.loc[coordenadas.CO_MUNICIPIO.eq(codigo) & coordenadas["elegivel_" + etapa]].copy()
        assert len(escola) > 0 and escola[campo].notna().all()
        demanda, taxa = criar_demanda(codigo, etapa, int(escola[campo].sum()))
        pares = salvar_instancia(codigo, etapa, escola, demanda)
        resumo.append({"codigo": codigo, "municipio": municipio, "etapa": etapa,
                       "setores": len(demanda), "setores_demanda_zero": int(demanda.demanda_inteira.eq(0).sum()),
                       "escolas": len(escola), "demanda": int(demanda.demanda_inteira.sum()),
                       "populacao": int(demanda.v0001.sum()), "idades_imputadas": int(demanda.idade_imputada.sum()),
                       "coordenadas_pendentes": int((~escola.apta).sum()), "taxa_imputacao": taxa,
                       "pares_previstos": len(escola) * len(demanda), "pares_exportados": pares})
resumo = pd.DataFrame(resumo)
resumo.to_csv(saida / "resumo_instancias.csv", index=False)
display(resumo)

,codigo,municipio,etapa,setores,setores_demanda_zero,escolas,demanda,populacao,idades_imputadas,coordenadas_pendentes,taxa_imputacao,pares_previstos,pares_exportados
0,3138203,Lavras,fundamental_iniciais,219,1,20,4453,104761,8,0,0.060183,4380,4380
1,3138203,Lavras,fundamental_finais,219,2,22,3636,104761,9,0,0.053558,4818,4818
2,3138203,Lavras,medio,219,2,8,2165,104761,10,0,0.066045,1752,1752
3,3143302,Montes Claros,fundamental_iniciais,885,30,87,22439,414240,52,5,0.066223,76995,0
4,3143302,Montes Claros,fundamental_finais,885,28,71,17448,414240,48,3,0.063884,62835,0
5,3143302,Montes Claros,medio,885,28,43,11370,414240,41,8,0.071337,38055,0
6,3136702,Juiz de Fora,fundamental_iniciais,1191,49,115,21467,540756,75,0,0.055257,136965,136965
7,3136702,Juiz de Fora,fundamental_finais,1191,49,100,19608,540756,68,0,0.054686,119100,119100
8,3136702,Juiz de Fora,medio,1191,52,36,11690,540756,58,2,0.060102,42876,0
9,3118601,Contagem,fundamental_iniciais,1312,25,81,27878,621863,40,0,0.061400,106272,106272


In [15]:
parametros = {"municipios": municipios, "cenario": "calibrada", "ibge": 2022, "inep": 2025,
              "setores": "todos os setores únicos da malha municipal, inclusive demanda zero",
              "amostragem": False, "tolerancia_divisa_catalogo_m": 250, "crs_coordenadas": "EPSG:4326", "distancia": "euclidiana em km",
              "escopo": "cinco maiores municípios de MG por população e Lavras",
              "capacidades": "ceil(alpha * matriculas), hipóteses sem vagas oficiais",
              "catalogo": "referência temporal não confirmada", "complemento_lavras": str(aprovadas)}
(saida / "parametros_preparacao.json").write_text(json.dumps(parametros, ensure_ascii=False, indent=2))
print("Instâncias completas preparadas em", saida)

Instâncias completas preparadas em /home/jz/PM/outputs/grandes_municipios
